# Interconnect Customer Churn Prediction

## Project Overview

Interconnect, a telecom operator, wants to identify customers who are likely to churn so its marketing 
team can proactively offer promotional codes and special plan options before those customers leave. This 
project builds a binary classification model to predict churn using contract, personal, internet, and 
phone service data collected across four separate datasets.

**Target:** `Churned`, derived from the `EndDate` column (`EndDate != 'No'` indicates a customer who has 
already churned).

**Primary metric:** AUC-ROC (target: ≥ 0.88)
**Secondary metric:** Accuracy

The notebook is organized into five phases:
1. **Merge & Consolidate** — combine the four source datasets into a single working DataFrame
2. **EDA** — explore feature distributions, class balance, and churn-rate patterns across categorical features
3. **Target & Feature Engineering** — derive the churn target and tenure feature, clean data quality issues, and encode categorical variables
4. **Model Training** — train and compare a logistic regression baseline against LightGBM and CatBoost
5. **Evaluation** — assess model quality, training/prediction speed, feature importance, and business implications

In [ ]:
import os
import pandas as pd
import datetime as dt
import time
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostClassifier

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve, RocCurveDisplay


# Phase 0 - Constants and Functions

In [ ]:
# Standardize the 'random_state' parameter across entire project
RND = 42

# Standardize which Scaler to use across the entire project
scaler = StandardScaler()

In [ ]:
# function to determine if csv file path exists locally or not
def self_detect(dataset):
    """
    Used to self-detect the path of the dataset.csv file
    """

    tripleten_path = '/datasets/final_provider/'

    local_path = '/Users/georgeknight/Desktop/TripleTen_Projects/Sprint 17 - Final Project/datasets/final_provider'

    # check if the path exists, if not, use the local path
    if os.path.exists(tripleten_path):
        dataset_path = os.path.join(tripleten_path, dataset)
    else:
        dataset_path = os.path.join(local_path, dataset)

    return dataset_path

In [ ]:
# function to loop over all categorical columns and calculate the churn rate
def churn_rate_by_category(df, column):
    """
    Calculates the churn rate based on a dataframe and a given column name.
    """

    summary = df_final.groupby(column)['Churned'].agg(['sum', 'count'])
    summary['churn_rate'] = round(summary['sum'] / summary['count'], 4)

    return summary

# Phase 1 - Load, Merge, Consolidate

### Step 1.1 - Load Datasets into DataFrames

In [ ]:
# loading all 4 datasets using 'self_detect' function
df_contract = pd.read_csv(self_detect('contract.csv'))
df_personal = pd.read_csv(self_detect('personal.csv'))
df_internet = pd.read_csv(self_detect('internet.csv'))
df_phone = pd.read_csv(self_detect('phone.csv'))

In [ ]:
df_contract.info()
df_personal.info()
df_internet.info()
df_phone.info()

### Step 1.2 - Merge 'df_contract' and 'df_personal'

In [ ]:
df_merged = df_contract.merge(df_personal, how='inner', on='customerID')

df_merged.info()

### Step 1.3 - Left join 'df_internet' and 'df_phone' -- filling missing rows explicitly with "NO Service"

In [ ]:
df_second_merge = df_merged.merge(df_internet, how='left', on='customerID')
df_final_merge = df_second_merge.merge(df_phone, how='left', on='customerID')

# fill missing rows from 'df_internet' and 'df_phone' with "No Service"
df_final = df_final_merge.fillna('No Service')

# Verify dtypes and rows match with no missing values
df_final.info()

In [ ]:
# filter 'TotalCharges' to only rows with blanks
filtered_df = df_final.loc[df_final['TotalCharges'] == ' ']
filtered_df['BeginDate'].unique()

# Resolve the 'TotalCharges' conversion from string to numeric
df_final['TotalCharges'] = pd.to_numeric(df_final['TotalCharges'], errors='coerce').fillna(0.00)
df_final.info()
#df_final['TotalCharges'].isna().sum()
print(df_final.head(10))

In [ ]:
print(len(filtered_df))

In [ ]:
sns.histplot(data=df_final, x='TotalCharges', binrange=(0, 500), bins=50)
plt.title('TotalCharges Distribution (Zoomed to $0-$500)')
plt.show()

#### Justification for Handling missing values in the Total Charges column
After filtering the dataframe to show just the rows with the blank values for Total Charges, it was found that there were 11 customers who had not yet had any charges to their account. This is mainly due to the fact their `StartDate` and the data `CutOffDate` were on the same day. 

Since there are only 11 customer's that needed to have the `TotalCharges` value filled in with `0.00`, this will not have any impact on skewing the distribution of this column. As is shown above in the histogram chart, the next few bins after the `0.00` bin register 80-150 in counted values.

In [ ]:
# Convert 'BeginDate' to dateime
df_final['BeginDate'] = pd.to_datetime(df_final['BeginDate'], errors='coerce', yearfirst=True, format='%Y-%m-%d')

# Create New 'EndDate' column and convert for calculation purposes only - preserves the original data
df_final['EndDate_dt'] = pd.to_datetime(df_final['EndDate'], errors='coerce', yearfirst=True, format='%Y-%m-%d %H:%M:%S')
df_final['EndDate_dt'] = df_final['EndDate_dt'].fillna(pd.Timestamp('2020-02-01'))

#df_final.info()
print(df_final.head(20))
df_final.info()


#### Justification for Using 2020-02-01 to Fill in Missing values for `EndDate` Column

The decision was made to fill the missing values in the `EndDate` column with the data cutoff date provided in the description for this project which was February 1, 2020. This was done by creating a new column named `EndDate_dt` so the original `EndDate` values would not be lost and could be used to determine which customer's in the dataset had already churned. The `EndDate_dt`values will be used later to calculate the `Tenure` of each contract to use as a feature for the models to look at.

# Phase 2 - EDA

In [ ]:
categorical_cols = ['Type', 'PaperlessBilling', 'PaymentMethod', 'gender', 'SeniorCitizen', 
                     'Partner', 'Dependents', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
                     'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'MultipleLines']

for col in categorical_cols:
    print(f"Number of Unique Values in {col} Column: ", df_final[col].nunique(), df_final[col].unique())

In [ ]:
# Derive 'tenure' of each client's contract
df_final['Tenure (months)'] = (df_final['EndDate_dt'] - df_final['BeginDate']).dt.days
df_final['Tenure (months)'] = round(df_final['Tenure (months)'] / 30.44, 3)

#print(df_final['Tenure (months)'])

In [ ]:
# Derive 'Churned' column to show clients who have already left
df_final['Churned'] = (df_final['EndDate'] != 'No').astype(int)

#print(df_final.head(20))
df_final['Churned'].value_counts()

In [ ]:
# Distribution of 'MonthlyCharges'
df_final['MonthlyCharges'].describe()

In [ ]:
# Distribution of 'TotalCharges'
df_final['TotalCharges'].describe()

In [ ]:
# Distribution of 'Tenure (months)'
df_final['Tenure (months)'].describe()

In [ ]:
# Histogram for 'MonthlyCharges' with 'Churned' overlay
fig, axes = plt.subplots(3, 1, figsize=[15,15])
sns.histplot(data=df_final,
             x='MonthlyCharges',
             hue='Churned',
             ax=axes[0])
axes[0].set_title('Distribution of Monthly Charges')
sns.histplot(data=df_final,
             x='TotalCharges',
             hue='Churned',
             ax=axes[1])
axes[1].set_title('Distribution of Total Charges')
sns.histplot(data=df_final,
             x='Tenure (months)',
             hue='Churned',
             ax=axes[2])
axes[2].set_title('Distribution of Tenure (months)')
plt.tight_layout()
plt.show()


In [ ]:
# Analyzing the Churn rate for categorical features
# Step A: get all object-dtype columns
categorical_cols = df_final.select_dtypes(include='object').columns

#Step B: drop columns with no meaningful information
categorical_cols = categorical_cols.drop(['customerID', 'EndDate'])

#Step C: Loop over remaining names, calling churn rate function on each
for col in categorical_cols:
    result = churn_rate_by_category(df_final, col)
    print(f"---{col}---")
    print(result)

### Contract Type Summary Conclusion

- Roughly a 15x difference in Churn Rate between 'Month-to-Month' & 'Two year' contracts
- 'Month-to-Month' contracts allow clients to leave at the end of any given month, where 'Two year' contracts lock a client into tougher process to leave in the middle of the contract.

### Senior Citizen Sumary

- A Senior Citizen is roughly 1.8x more likely to leave then a non-Senior Citizen

### Internet Service Summary
- Fiber Optic clients are over 5x as likely to churn than those without Interconnect's internet services.
- This is counter-intuitive since Fiber Optics is typically a premium service
- Begs the question: Is it a quality problem? Is it a pricing problem? Is it something else entirely? However, This is a question that can't be answered with the limited information that was given for this project, but would be something I would mention as a secondary aspect for Interconnect Telecom to take a look into at thier leisure.

### Multiple Lines Summary
- The spread here is less than 5 points
- This doesn't seem to have that large of an impact on the churn rate

### Payment Method Summary
- Electronic Check payment method is almost 3x higher than the other 3 payment methdods

# Phase 3 - Target & Feature Engineering

In [ ]:
# Derive 'HasInternet' column to show clients who have Internet Services
df_final['HasInternet'] = (df_final['InternetService'] != 'No Service').astype(int)

#print(df_final.head(20))
df_final['HasInternet'].value_counts()

In [ ]:
# List of 6 columns linked ot 'HasInternet'
internet_services = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

# Loop over 6 columns and replace 'No Service' with 'No'
for i in internet_services:
    df_final[i] = df_final[i].replace('No Service', 'No')
    

In [ ]:
# Derive 'HasPhone' column to show clients who have Phone Services
df_final['HasPhone'] = (df_final['MultipleLines'] != 'No Service').astype(int)

# Clean 'MultipleLines' column to only have 'Yes' or 'No' values
df_final['MultipleLines'] = df_final['MultipleLines'].replace('No Service', 'No')

df_final['MultipleLines'].value_counts()

In [ ]:
# Using pd.get_dummies for 'PaymentMethod' feature
payment_dummies = pd.get_dummies(df_final['PaymentMethod'], prefix='PaymentMethod', drop_first=True)
payment_dummies = payment_dummies.astype(int)

df_final = pd.concat([df_final, payment_dummies], axis=1).drop(columns=['PaymentMethod'])

#print(df_final.head(10))

In [ ]:
# Filtering out 'No Service' from 'InternetService' column for dummies creation
df_final['InternetService'] = df_final['InternetService'].replace('No Service', np.nan)

# Using pd.get_dummies for 'InternetService' feature
internet_dummies = pd.get_dummies(df_final['InternetService'], prefix='InternetService', drop_first=True, dummy_na=False)
internet_dummies = internet_dummies.astype(int)

df_final = pd.concat([df_final, internet_dummies], axis=1).drop(columns=['InternetService'])

#print(df_final.head(10))
#df_final.info()


In [ ]:
# filter to verify that all dummies have been created and no 'No Service' values remain
df_final_filtered = df_final[df_final['HasInternet'] == 0]

#print(len(df_final))
#print(df_final_filtered.head(20))

In [ ]:
# Using Ordinal Encoding for 'Type' feature
type_mapping = {'Month-to-month': 0, 'One year': 1, 'Two year': 2}
df_final['Type'] = df_final['Type'].map(type_mapping)

#df_final['Type'].value_counts()

In [ ]:
# Using Ordinal Encoding for 'gender' feature
gender_mapping = {'Female': 0, 'Male': 1}
df_final['gender'] = df_final['gender'].map(gender_mapping)


In [ ]:
# Looping through the categorical columns to create a binary mapping dictionary for each column
cat_col_list = ['PaperlessBilling', 'Partner', 'Dependents', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'MultipleLines']
cat_col_mapping = {'Yes': 1, 'No': 0}

for col in cat_col_list:
    df_final[col] = df_final[col].map(cat_col_mapping)


In [ ]:
#df_final.info()

In [ ]:
# Dropping irrelevant columns
df_model_ready = df_final.drop(columns=['customerID', 'BeginDate', 'EndDate', 'EndDate_dt'])

# Check for duplicates
duplicate_count = df_model_ready.duplicated().sum()
print(duplicate_count)

In [ ]:
df_model_ready = df_model_ready.drop_duplicates(keep='first')
print(df_model_ready.shape)

In [ ]:
# Building the features and target variable for the model
features = df_model_ready.drop(columns=['Churned'], axis=1)
target = df_model_ready['Churned']


In [ ]:
# Splitting the merged dataset into Training set of 60% and Validation/Test set of 40%
X_train, X_val_test, y_train, y_val_test = train_test_split(features, target, test_size=0.4, random_state=RND, stratify=target)

y_train.value_counts(normalize=True)

In [ ]:
# Splitting the Val/Test Set into Validation set of 20% and Test set of 20%
X_val, X_test, y_val, y_test = train_test_split(X_val_test, y_val_test, test_size=0.5, random_state=RND, stratify=y_val_test)

In [ ]:

y_test.value_counts(normalize=True)

In [ ]:
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)


# Phase 4 - Model Training

In [ ]:
results = []

### Logistic Regression

In [ ]:
# Scale the features using StandardScaler
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Baseline Model - Logistic Regression with class_weight='balanced' to handle class imbalance
training_start_time = time.time()
model_0 = LogisticRegression(class_weight='balanced', random_state=RND)
model_0.fit(X_train_scaled,y_train)
model_0_time = round(time.time() - training_start_time, 4)


pred_start_time = time.time()
y_pred_0 = model_0.predict(X_val_scaled)
y_proba_0 = model_0.predict_proba(X_val_scaled)[:, 1]
model_0_pred_time = round(time.time() - pred_start_time, 4)

# Evaluate the model performance using accuracy and ROC AUC score
auc_0 = round(roc_auc_score(y_val, y_proba_0), 4)
acc_0 = round(accuracy_score(y_val, y_pred_0), 4)

results.append({'Model': 'Logistic Regression', 'AUC': auc_0, 'Accuracy': acc_0, 'Training Time': model_0_time, 'Prediction Time': model_0_pred_time})

print(f"Model 0 - AUC: {auc_0}, Accuracy: {acc_0}, Training Time: {model_0_time}, Prediction Time: {model_0_pred_time}")

### LightGBM

In [ ]:
# LightGBM Model with is_unbalance=True to handle class imbalance
training_start_time = time.time()
model_1 = lgb.LGBMClassifier(is_unbalance=True, random_state=RND)
model_1.fit(X_train, y_train)
model_1_time = round(time.time() - training_start_time, 4)

pred_start_time = time.time()
y_pred_1 = model_1.predict(X_val)
y_proba_1 = model_1.predict_proba(X_val)[:, 1]
model_1_pred_time = round(time.time() - pred_start_time, 4)

# Evaluate the model performance using accuracy and ROC AUC score
auc_1 = round(roc_auc_score(y_val, y_proba_1), 4)
acc_1 = round(accuracy_score(y_val, y_pred_1), 4)

results.append({'Model': 'LightGBM', 'AUC': auc_1, 'Accuracy': acc_1, 'Training Time': model_1_time, 'Prediction Time': model_1_pred_time})

print(f"Model 1 - AUC: {auc_1}, Accuracy: {acc_1}, Training Time: {model_1_time}, Prediction Time: {model_1_pred_time}")

### CatBoost

In [ ]:
# CatBoost
training_start_time = time.time()
model_2 = CatBoostClassifier(auto_class_weights='Balanced', random_state=RND, verbose=False)
model_2.fit(X_train, y_train)
model_2_time = round(time.time() - training_start_time, 4)

pred_start_time = time.time()
y_pred_2 = model_2.predict(X_val)
y_proba_2 = model_2.predict_proba(X_val)[:, 1]
model_2_pred_time = round(time.time() - pred_start_time, 4)

# Evaluate the model performance using accuracy and ROC AUC score
auc_2 = round(roc_auc_score(y_val, y_proba_2), 4)
acc_2 = round(accuracy_score(y_val, y_pred_2), 4)

results.append({'Model': 'CatBoost', 'AUC': auc_2, 'Accuracy': acc_2, 'Training Time': model_2_time, 'Prediction Time': model_2_pred_time})

print(f"Model 2 - AUC: {auc_2}, Accuracy: {acc_2}, Training Time: {model_2_time}, Prediction Time: {model_2_pred_time}")

In [ ]:
results_df = pd.DataFrame(results)

results_df

# Phase 5 - Evaluation

In [ ]:
# Empty list to collect evaluation results for chosen model
final_results = []

In [ ]:
# Evaluation of the chosen model (LightGBM) on the Test Set
eval_start_time = time.time()
y_pred_final = model_1.predict(X_test)
y_proba_final = model_1.predict_proba(X_test)[:, 1]
eval_time = round(time.time() - eval_start_time, 4)

# Evaluate the model performance using accuracy and ROC AUC score
auc_final = round(roc_auc_score(y_test, y_proba_final), 4)
acc_final = round(accuracy_score(y_test, y_pred_final), 4)

final_results.append({'Model': 'LightGBM', 'Eval AUC': auc_final, 'Eval Accuracy': acc_final, 'Evaluation Time': eval_time})

print(f"Eval of LightGBM MOdel - AUC: {auc_final}, Accuracy: {acc_final}, Eval Prediction Time: {eval_time}")

In [ ]:
results_df = pd.DataFrame(results)
eval_results_df = pd.DataFrame(final_results)
final_results_df = pd.merge(results_df, eval_results_df, on='Model', how='left')

final_results_df

In [ ]:
# Confusion Matrix for the best performing model (LightGBM)
cm = confusion_matrix(y_test, y_pred_1)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title('LightGBM Confusion Matrix')
plt.show()

#### Confusion Matrix Interpretation

- Top-Left corner shows 709 customers who stayed and model correctly predicted to stay. (True Negatives)
- Top-Righ corner shows 324 customers who stayed, but model predicted they would churn (False Positives)
- Bottom-Left shows 251 customers who actually churned, but the model predicted they would stay. (False Negatives)
- Bottom-Right shows 121 customers who churned and the model correctly predicted they would churn.(True Positives)

Based on this data, Interconnect could potentially miss 251 customers who are about to churn and leave the company without Interconnect reaching out to offer them a "Promo code" or some other kind of savings. However, the company could also spend money reaching out to the 324 customers the model indicates are going to churn who actually had no thoughts of leaving the company. So this begs the question "Which side would cost the company more money...Not reaching out the the 251 before they leave or reaching out to the 324 because we think they might leave?

- The **Recall** of the churned class is `121 / 372 = 32.53%`
- The **Precision** of the churned class is `121 / 445 = 27.19%`

**Note**
- Recall is the number of True Positives divided by the Total number of Customer's Churned
- Precision is the number of True Positives divided by the Total number the Model Flagged as likely to Churn

#### Final Model Selection & SP Threshold

The best-performing model was **LightGBM**, achieving an Final AUC-ROC of **0.8903** and Final Accuracy of **81.99%** on the `Test set`. 

Per the project's evaluation criteria, an AUC-ROC ≥ 0.88 corresponds to the maximum score tier of **6 SP**.


In [ ]:
# ROC Curve for the best performing model (LightGBM)
fpr_test, tpr_test, thresholds_test = roc_curve(y_test, y_proba_final)
plt.plot(fpr_test, tpr_test, label='LightGBM Test (AUC = {auc_final:.4f})')
plt.plot([0, 1], [0, 1], linestyle='--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('LightGBMROC Curve')
plt.legend(loc='lower right')
plt.show()

In [ ]:
# Feature Importance for the best performing model (LightGBM)
feature_importance = pd.DataFrame({'Feature': features.columns, 'Importance': model_1.feature_importances_})
feature_importance = feature_importance.sort_values(by='Importance', ascending=False)

feature_importance

#### Feature Importance — Interpretation Notes

The LightGBM feature importance ranking shows `Tenure (months)` as the dominant feature by a wide margin, followed by `MonthlyCharges` and `TotalCharges`. However, this ranking needs to be read alongside the EDA findings rather than taken at face value on its own.

**`Type` ranks lower than expected.** During EDA, `Type` (contract length) showed the strongest churn-rate spread of any feature — 42.7% Month-to-month vs 2.8% Two-year. Despite this, it ranks well below `Tenure (months)` in feature importance. This is likely explained by the relationship contract-type and tenure share together. For example: a Month-to-month customer has the opportunity to end a contract at the end of every month with very little issue; however, a Two-year customer typically has to deal with paying to get out of their contract and thus this tends to cause those customers to stick around longer. Because LightGBM found `tenure` to be the more flexible, finer-grained feature for capturing this signal, it likely "absorbed" much of the predictive value that would otherwise be attributed to `Type`. This is a known consequence of mulitcollinearity or feature overlap that was flagged earlier in EDA — it affects how credit is distributed across correlated features without necessarily hurting the model's overall predictive accuracy.

**`gender` ranks higher than expected.** EDA showed 26.92% vs 26.16% churn rate, indicating essentially no real relationship between gender and churn. Its appearance mid-table in feature importance does not contradict this — feature importance measures how often/usefully a feature was used across tree splits, not how strongly it correlates with the target on its own. With this being said, it is important to remember that a feature can pick up minor importance from noise or interaction effects even when it has no true meaningful standalone signal. For this reason, the EDA churn-rate comparison remains the more reliable and interpretable source for business conclusions about `gender`'s relevance.

**Takeaway:** feature importance from a tree ensemble is most useful for confirming overall predictive power, but EDA-level churn-rate analysis remains the clearer tool for identifying which individual business factors most directly drive churn — especially when features are correlated with one another.

## Conclusion

### Project Summary
This project set out to predict customer churn for Interconnect, a telecom operator, using contract, personal, internet, and phone service data for `7043` customers. The target was derived from the `EndDate` column, with churn defined as any customer whose contract had an end date on record. Given a class imbalance of roughly `73.5%/26.5%` retained-to-churned, AUC-ROC was used as the primary evaluation metric, with Accuracy as a secondary measure.

### Data Preparation
Four datasets were merged on `customerID`. Key preparation steps included:
- Cleaning `TotalCharges` (originally stored as an object dtype due to blank strings for new customers with zero elapsed billing periods)
- Deriving `tenure` from `BeginDate` and `EndDate`, using the data's snapshot date (February 1, 2020) as the reference point for still-active customers
- Collapsing redundant "No Service" categories across six internet add-on columns and `MultipleLines` into dedicated `HasInternet`/`HasPhone` flags, to avoid the same fact being duplicated across multiple encoded columns
- Encoding categorical features using strategy matched to each feature's structure: ordinal encoding for `Type` (given its inherent order), one-hot/dummy encoding for low-cardinality unordered categories, and manual binary mapping for two-category features

### Exploratory Findings
EDA identified `Type`, `InternetService`, `PaymentMethod`, `SeniorCitizen` as the strongest individual churn signals, with churn rates ranging from `2.83%` to `42.71%` depending on contract type alone. `gender` and `MultipleLines` showed negligible predictive value on their own. Cross-tabulation also revealed meaningful overlap between `Type` and `PaymentMethod` — [Month-to-month customers disproportionately paid via Electronic check] — suggesting these features may partially capture the same underlying "disengaged customer" pattern rather than acting as fully independent signals.

### Model Comparison

| Model | AUC-ROC | Accuracy | Training Time (s) | Prediction Time (s) |  Eval AUC-ROC | Eval Accuracy | Evaluation Time (s) |
|---|---|---|---|---|---|---|---|
| Logistic Regression | 0.8550 | 0.7409 | 0.0388 | 0.0004 | NaN | NaN | NaN |
| LightGBM | 0.8879 | 0.8171 | 0.3063 | 0.0069 | 0.8903 | 0.8199 | 0.0173 |
| CatBoost | 0.8913 | 0.8171 | 1.7662 | 0.0032 | NaN | NaN | NaN |

**LightGBM was selected as the final model**, achieving the second highest AUC-ROC `(0.8879)` — clearing the project's ≥0.88 threshold for the maximum score tier `(6 SP)` — while also training almost `6x` faster than CatBoost, which achieved comparable but slightly higher quality `(0.8913 AUC)` but at a significantly higher training cost. Logistic regression, while fastest, produced the weakest separation between classes.

### Feature Importance vs. EDA
The `Tenure (months)` feature dominated teh importance ranking likely absorbing the signal that found during EDA that was attridbuted to `Type` due to their structural overlap. The higher ranekd importance of `gender` (compared to near zero during EDA) illsutrates that tree-based feature importance reflects split usage rather than standalone predicitve strengths.

### Business Recommendation
The final model's confusion matrix showed `251` missed churners (false negatives) against `324` customers incorrectly flagged as at-risk (false positives). Given that a missed churner represents a fully lost customer while a false positive only costs a modest promotional incentive, My recommendation is the Interconnect Marketing team should favor a model or decision threshold that prioritizes recall over precision for the churned class — the cost of a missed at-risk customer outweighs the cost of an unnecessary retention offer.